In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


ModuleNotFoundError: No module named 'kagglehub'

In [2]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

avenir_hku_web_path = kagglehub.competition_download('avenir-hku-web')

print('Data source import complete.')


NameError: name 'kagglehub' is not defined

In [ ]:
import numpy as np
import pandas as pd
pd.options.mode.copy_on_write = True
import datetime, os, time
import multiprocessing as mp
from sklearn.linear_model import LinearRegression

import lightgbm as lgb
from math import log

import glob
import gc

In [ ]:
class OlsModel:
    def __init__(self):
        self.train_data_path = "/kaggle/input/avenir-hku-web/kline_data/train_data"
        self.start_datetime = datetime.datetime(2021, 3, 1, 0, 0, 0)

        self.SUBMISSION_MODE = False
        self.VERBOSE_PROGRESS = not self.SUBMISSION_MODE

        self.MAX_WORKERS = max(1, min(8, mp.cpu_count()-1))
        self.REF_SYMBOL = None
        self.DEV_MAX_SYMBOLS = 0
        self.DEV_LAST_MONTHS = 0
        self.USE_CACHE = True
        self.CACHE_PATH = "/kaggle/working/panel_cache_npz.npz"

        self.windows_main = [8, 24, 48, 96, 288, 672, 1344]
        self.winsor_q = 0.01
        self.topk_features = 0
        self.use_time_decay = True
        self.halflife_days = 90
        self.use_early_stopping = True
        self.num_threads = max(1, mp.cpu_count()-1)
        self.num_boost_round = 2000
        self.es_rounds = 200
        self.lr = 0.05
        self.valid_last_days = 60

    def _log(self, msg):
        if self.VERBOSE_PROGRESS:
            print(msg, flush=True)

    def _tic(self):
        return time.time()

    def _toc(self, t0, label=""):
        if self.VERBOSE_PROGRESS:
            dt = time.time() - t0
            print(f"[time] {label} took {dt:.2f}s", flush=True)

    """I/O helpers"""
    def get_all_symbol_list(self):
        parquet_name_list = os.listdir(self.train_data_path)
        symbol_list = [parquet_name.split(".")[0] for parquet_name in parquet_name_list]
        return symbol_list

    def get_single_symbol_kline_data(self, symbol):
        try:
            df = pd.read_parquet(f"{self.train_data_path}/{symbol}.parquet")
            df = df.set_index("timestamp")
            df = df.astype(np.float64)
            df['vwap'] = (df['amount'] / df['volume']).replace([np.inf, -np.inf], np.nan).ffill()
        except Exception as e:
            self._log(f"get_single_symbol_kline_data error: {e}")
            df = pd.DataFrame()
        return df

    def get_all_symbol_kline(self):
        def _log(msg):
            if self.VERBOSE_PROGRESS:
                print(msg, flush=True)

        t0 = time.time()
        files = [f for f in os.listdir(self.train_data_path) if f.endswith(".parquet")]
        files.sort()
        total = len(files)
        if total == 0:
            raise FileNotFoundError("No parquet files found in train_data_path")

        """Dev subset"""
        if self.DEV_MAX_SYMBOLS and self.DEV_MAX_SYMBOLS > 0:
            files = files[:self.DEV_MAX_SYMBOLS]
        _log(f"[load] found {total} parquet files; using {len(files)} after DEV_MAX_SYMBOLS")

        """Cache fast path with coverage check"""
        if self.USE_CACHE and os.path.exists(self.CACHE_PATH):
            _log(f"[cache] loading cached panel from {self.CACHE_PATH} ...")
            dat = np.load(self.CACHE_PATH, allow_pickle=True)
            time_arr = dat["time_arr"]
            cache_dt = pd.to_datetime(time_arr)  # datetime64
            cache_min = cache_dt.min()
            cache_max = cache_dt.max()
            if cache_min <= pd.Timestamp(self.start_datetime):
                open_price_arr  = dat["open"]
                high_price_arr  = dat["high"]
                low_price_arr   = dat["low"]
                close_price_arr = dat["close"]
                vwap_arr        = dat["vwap"]
                amount_arr      = dat["amount"]
                symbols         = list(dat["symbols"])
                _log(f"[cache] OK: covers {cache_min} .. {cache_max}")
                _log(f"[time] get_all_symbol_kline (cache) took {time.time()-t0:.2f}s")
                return symbols, time_arr, open_price_arr, high_price_arr, low_price_arr, close_price_arr, vwap_arr, amount_arr
            else:
                _log(f"[cache] SKIP: cache starts at {cache_min}, later than required {self.start_datetime}. Rebuilding...")

        """Build from parquet with a robust reference index"""
        COLS = ["timestamp","open_price","high_price","low_price","close_price","volume","amount"]

        """Pick a reference file by EARLIEST start timestamp among all files"""
        ref_file = None
        ref_start = None
        for f in files:
            p = os.path.join(self.train_data_path, f)
            try:
                ts = pd.read_parquet(p, columns=["timestamp"])["timestamp"]
                if ts.empty:
                    continue
                smin = ts.min()
                if (ref_start is None) or (smin < ref_start):
                    ref_start = smin
                    ref_file = f
            except Exception as e:
                _log(f"[load][warn] failed min-ts scan for {f}: {e}")
                continue

        if ref_file is None:
            """fallback to alphabetical first if scanning completely failed"""
            ref_file = files[0]

        self.REF_SYMBOL = os.path.splitext(ref_file)[0]
        _log(f"[load] reference index = {self.REF_SYMBOL} (earliest start @ {pd.to_datetime(ref_start, unit='ms') if ref_start is not None else 'unknown'})")

        """Read reference file (once)"""
        ref = (pd.read_parquet(os.path.join(self.train_data_path, ref_file), columns=COLS)
                 .set_index("timestamp").astype(np.float64))
        ref["vwap"] = (ref["amount"]/ref["volume"]).replace([np.inf,-np.inf], np.nan).ffill()
        ref_idx = ref.index
        idx_dt = pd.to_datetime(ref_idx, unit="ms")

        """Optional dev trim"""
        if self.DEV_LAST_MONTHS and self.DEV_LAST_MONTHS > 0:
            cutoff = idx_dt.max() - pd.DateOffset(months=self.DEV_LAST_MONTHS)
            keep_mask = idx_dt >= cutoff
            ref = ref.loc[keep_mask]
            ref_idx = ref.index
            idx_dt = pd.to_datetime(ref_idx, unit="ms")
            _log(f"[load] DEV_LAST_MONTHS={self.DEV_LAST_MONTHS} → kept {keep_mask.sum()} bars")

        """Prepare arrays (T × N)"""
        N = len(files)
        T = len(ref_idx)
        open_price_arr  = np.full((T, N), np.nan, dtype=np.float32)
        high_price_arr  = np.full((T, N), np.nan, dtype=np.float32)
        low_price_arr   = np.full((T, N), np.nan, dtype=np.float32)
        close_price_arr = np.full((T, N), np.nan, dtype=np.float32)
        vwap_arr        = np.full((T, N), np.nan, dtype=np.float32)
        amount_arr      = np.full((T, N), np.nan, dtype=np.float32)

        """Fill reference column first (col 0)"""
        open_price_arr[:,0]  = ref["open_price"].reindex(ref_idx).to_numpy(np.float32)
        high_price_arr[:,0]  = ref["high_price"].reindex(ref_idx).to_numpy(np.float32)
        low_price_arr[:,0]   = ref["low_price"].reindex(ref_idx).to_numpy(np.float32)
        close_price_arr[:,0] = ref["close_price"].reindex(ref_idx).to_numpy(np.float32)
        vwap_arr[:,0]        = ref["vwap"].reindex(ref_idx).to_numpy(np.float32)
        amount_arr[:,0]      = ref["amount"].reindex(ref_idx).to_numpy(np.float32)

        symbols = [os.path.splitext(ref_file)[0]] + [os.path.splitext(f)[0] for f in files if f != ref_file]

        """Parallel read remaining files"""
        from multiprocessing.pool import ThreadPool
        pool = ThreadPool(self.MAX_WORKERS)
        pending = []

        def _read_one(path):
            try:
                df = pd.read_parquet(path, columns=COLS).set_index("timestamp").astype(np.float64)
                df["vwap"] = (df["amount"]/df["volume"]).replace([np.inf,-np.inf], np.nan).ffill()
                return df
            except Exception as e:
                return e

        for f in files:
            if f == ref_file:
                continue
            pending.append((f, pool.apply_async(_read_one, (os.path.join(self.train_data_path, f), ))))
        pool.close()

        col = 1
        for k, (fname, fut) in enumerate(pending, start=1):
            if k % 10 == 0 or k <= 5:
                _log(f"[load] reading file {k}/{len(pending)}: {fname}")
            result = fut.get()
            if isinstance(result, Exception):
                _log(f"[load][warn] failed to read {fname}: {result}")
                continue
            df = result.reindex(ref_idx)  # align to EARLY reference grid
            open_price_arr[:,col]  = df["open_price" ].to_numpy(np.float32, copy=False)
            high_price_arr[:,col]  = df["high_price" ].to_numpy(np.float32, copy=False)
            low_price_arr[:,col]   = df["low_price"  ].to_numpy(np.float32, copy=False)
            close_price_arr[:,col] = df["close_price"].to_numpy(np.float32, copy=False)
            vwap_arr[:,col]        = df["vwap"       ].to_numpy(np.float32, copy=False)
            amount_arr[:,col]      = df["amount"     ].to_numpy(np.float32, copy=False)
            col += 1
        pool.join()

        time_arr = idx_dt.values

        """Save (fresh) cache"""
        if self.USE_CACHE:
            try:
                np.savez_compressed(
                    self.CACHE_PATH,
                    time_arr=time_arr,
                    open=open_price_arr,
                    high=high_price_arr,
                    low=low_price_arr,
                    close=close_price_arr,
                    vwap=vwap_arr,
                    amount=amount_arr,
                    symbols=np.array(symbols, dtype=object)
                )
                _log(f"[cache] saved merged panel to {self.CACHE_PATH}")
            except Exception as e:
                _log(f"[cache][warn] failed saving cache: {e}")

        _log(f"[time] get_all_symbol_kline took {time.time()-t0:.2f}s")
        return symbols, time_arr, open_price_arr, high_price_arr, low_price_arr, close_price_arr, vwap_arr, amount_arr

    """RAM-SAFE HELPERS (add to class)"""
    def _precompute_ema_panels(self, df_vwap):
        ema_panels = {}
        for w in self.windows_main:
            ema_panels[w] = df_vwap.ewm(span=w, min_periods=w).mean().astype(np.float32)
        return ema_panels

    def _build_y_and_groups(self, df_target, start_dt, time_index, max_w):
        y_long = df_target.stack().rename("target")
        y_long = y_long.loc[y_long.index.get_level_values(0) >= start_dt].dropna().sort_index()

        """keep only rows where we have at least w history in base index"""
        tpos = pd.Index(time_index).get_indexer(y_long.index.get_level_values(0))
        ok = (tpos >= max(self.windows_main))
        if (~ok).any():
            y_long = y_long[ok]
            tpos = tpos[ok]

        gsize = y_long.groupby(level=0).size().astype(int)
        group_sizes = gsize.values.tolist()
        group_times = gsize.index.to_numpy()
        return y_long, np.asarray(group_sizes, dtype=np.int32), group_times

    def _fill_memmap_features(self, X_mm, col0, df_vwap, ema_panels, y_long, group_sizes, group_times, symbols):
        sym2col = {s:i for i,s in enumerate(symbols)}
        vwap = df_vwap.to_numpy(copy=False)  # (T,N)
        idx  = df_vwap.index
        N    = len(symbols)

        offsets = np.cumsum([0] + list(group_sizes[:-1]))
        y_idx = y_long.index   # MultiIndex (time, symbol)

        for w in self.windows_main:
            ema_w = ema_panels[w].to_numpy(copy=False)
            col = col0

            for t, gsz, off in zip(group_times, group_sizes, offsets):
                sl = slice(off, off+gsz)
                syms = y_idx[sl].get_level_values(1).to_numpy()
                cols_idx = np.fromiter((sym2col[s] for s in syms), count=gsz, dtype=np.int32)

                tpos = idx.get_loc(t)
                vt   = vwap[tpos, :]
                vtm  = vwap[tpos - w, :]
                ema_t= ema_w[tpos, :]

                mom_all    = vt / vtm - 1.0
                emagap_all = vt / ema_t - 1.0

                """cross-sectional mean/std (nan-safe)"""
                m_mom = np.nanmean(mom_all); s_mom = np.nanstd(mom_all, ddof=1); s_mom = s_mom if np.isfinite(s_mom) and s_mom!=0 else np.nan
                m_ema = np.nanmean(emagap_all); s_ema = np.nanstd(emagap_all, ddof=1); s_ema = s_ema if np.isfinite(s_ema) and s_ema!=0 else np.nan

                """ranks across ALL symbols; normalize by N"""
                r_mom_all = pd.Series(mom_all).rank(ascending=False, method="average").to_numpy() / float(N)
                r_ema_all = pd.Series(emagap_all).rank(ascending=False, method="average").to_numpy() / float(N)

                mom      = mom_all[cols_idx]
                emagap   = emagap_all[cols_idx]
                momdm    = mom - m_mom
                emagapdm = emagap - m_ema
                momz     = (mom - m_mom) / s_mom if np.isfinite(s_mom) else np.zeros_like(mom)
                emagapz  = (emagap - m_ema) / s_ema if np.isfinite(s_ema) else np.zeros_like(emagap)
                r_mom    = r_mom_all[cols_idx]
                r_ema    = r_ema_all[cols_idx]

                X_mm[sl, col+0] = mom.astype(np.float32)
                X_mm[sl, col+1] = momdm.astype(np.float32)
                X_mm[sl, col+2] = momz.astype(np.float32)
                X_mm[sl, col+3] = emagap.astype(np.float32)
                X_mm[sl, col+4] = emagapdm.astype(np.float32)
                X_mm[sl, col+5] = emagapz.astype(np.float32)
                X_mm[sl, col+6] = r_mom.astype(np.float32)
                X_mm[sl, col+7] = r_ema.astype(np.float32)

            col0 += 8

        return col0

    def _fill_memmap_seasonality(self, X_mm, col0, y_long):
        t = y_long.index.get_level_values(0)
        hours = t.hour.to_numpy()
        dows  = t.dayofweek.to_numpy()
        X_mm[:, col0+0] = np.sin(2*np.pi*hours/24.0).astype(np.float32)
        X_mm[:, col0+1] = np.cos(2*np.pi*hours/24.0).astype(np.float32)
        X_mm[:, col0+2] = np.sin(2*np.pi*dows /7.0).astype(np.float32)
        X_mm[:, col0+3] = np.cos(2*np.pi*dows /7.0).astype(np.float32)
        return col0 + 4


    def _load_submission_id(self):
        """Return (df, path) and normalize to an 'id' column."""
        cands = [
            "submission_id.csv",
            "/kaggle/input/*/submission_id.csv",
            "/kaggle/input/**/submission_id.csv",
            "/kaggle/input/testing-for-crypto-3/submission_id.csv",
        ]
        for pat in cands:
            hits = glob.glob(pat, recursive=True)
            if hits:
                path = hits[0]
                try:
                    df = pd.read_csv(path)
                except Exception:
                    """handle headerless file: single column with no name"""
                    df = pd.read_csv(path, header=None)
                    df.columns = ["id"]
                """normalize col name"""
                if "id" not in df.columns:
                    # pick the first column
                    df = df.rename(columns={df.columns[0]: "id"})
                """ensure it's string"""
                df["id"] = df["id"].astype(str)
                return df[["id"]].copy(), path
        raise FileNotFoundError("submission_id.csv not found under /kaggle/input or CWD")

    """Metric"""
    def weighted_spearmanr(self, y_true, y_pred):
        n = len(y_true)
        r_true = pd.Series(y_true).rank(ascending=False, method='average')
        r_pred = pd.Series(y_pred).rank(ascending=False, method='average')
        x = 2 * (r_true - 1) / (n - 1) - 1
        w = x ** 2
        w_sum = w.sum()
        mu_true = (w * r_true).sum() / w_sum
        mu_pred = (w * r_pred).sum() / w_sum
        cov = (w * (r_true - mu_true) * (r_pred - mu_pred)).sum()
        var_true = (w * (r_true - mu_true)**2).sum()
        var_pred = (w * (r_pred - mu_pred)**2).sum()
        return cov / np.sqrt(var_true * var_pred)

    """Feature helpers"""
    def _ema(self, df, span):
        return df.ewm(span=span, min_periods=span).mean()

    def _zscore_cs(self, df):
        m = df.mean(axis=1)
        s = df.std(axis=1).replace(0, np.nan)
        return (df.sub(m, axis=0)).div(s, axis=0)

    def _demean_cs(self, df):
        return df.sub(df.mean(axis=1), axis=0)

    def _parkinson_bar(self, high, low):
        x = (np.log(high / low)) ** 2
        return x / (4.0 * np.log(2.0))

    def _csr(self, df):
        return df.rank(axis=1, method="average", ascending=False).div(df.shape[1])

    def _winsorize_per_ts(self, s, q=0.01):
        out = []
        for t, g in s.groupby(level=0):
            lo, hi = g.quantile(q), g.quantile(1 - q)
            out.append(g.clip(lo, hi))
        return pd.concat(out).sort_index()

    def _neutralize_by_amount(self, preds_series, df_amount_long):
        df = pd.concat([preds_series.rename("p"), df_amount_long.rename("amt")], axis=1).fillna(0)
        outs = []
        for t, g in df.groupby(level=0):
            w = g["amt"].values
            w = np.where(w <= 0, 0.0, w)
            if w.sum() == 0:
                adj = g["p"] - g["p"].mean()
            else:
                mu = np.average(g["p"].values, weights=w)
                adj = g["p"] - mu
            outs.append(adj)
        return pd.concat(outs).sort_index()

    def _build_features(self, df_vwap, df_high, df_low, df_amount, idx, symbols):
        self._log("[FE] building base series...")
        t0 = self._tic()
        r1 = df_vwap.pct_change(1, fill_method=None)
        hlp = self._parkinson_bar(df_high, df_low)
        vol_proxy = (df_amount / df_vwap).replace([np.inf, -np.inf], np.nan)

        feats = {}

        for w in self.windows_main:
            mom = df_vwap / df_vwap.shift(w) - 1
            ema = self._ema(df_vwap, w)
            emagap = df_vwap / ema - 1
            volw = r1.rolling(w).std()
            pkw = hlp.rolling(w).mean()
            tostd = vol_proxy.pct_change(fill_method=None).rolling(w).std()

            feats[f"mom_{w}"] = mom.astype(np.float32)
            feats[f"momdm_{w}"] = self._demean_cs(mom).astype(np.float32)
            feats[f"momz_{w}"] = self._zscore_cs(mom).astype(np.float32)

            feats[f"emagap_{w}"] = emagap.astype(np.float32)
            feats[f"emagapdm_{w}"] = self._demean_cs(emagap).astype(np.float32)
            feats[f"emagapz_{w}"] = self._zscore_cs(emagap).astype(np.float32)

            feats[f"vol_{w}"] = volw.astype(np.float32)
            feats[f"pk_{w}"] = pkw.astype(np.float32)
            feats[f"tostd_{w}"] = tostd.astype(np.float32)

            feats[f"rank_mom_{w}"] = self._csr(mom).astype(np.float32)
            feats[f"rank_emagap_{w}"] = self._csr(emagap).astype(np.float32)

        self._log("[FE] adding seasonality...")
        hours = pd.Index(idx).hour.values
        dows = pd.Index(idx).dayofweek.values
        sin_h = np.sin(2 * np.pi * hours / 24)[:, None]
        cos_h = np.cos(2 * np.pi * hours / 24)[:, None]
        sin_d = np.sin(2 * np.pi * dows / 7)[:, None]
        cos_d = np.cos(2 * np.pi * dows / 7)[:, None]
        for name, mat in [("sin_h", sin_h), ("cos_h", cos_h), ("sin_d", sin_d), ("cos_d", cos_d)]:
            feats[name] = pd.DataFrame(np.repeat(mat, len(symbols), axis=1), index=idx, columns=symbols).astype(np.float32)

        self._toc(t0, "feature engineering")
        return feats

    def _stack_long(self, panels, y=None, start_dt=None):
        keys = sorted(panels.keys())
        long_list = []
        for k in keys:
            tmp = panels[k].stack()
            tmp.name = k
            long_list.append(tmp)
        X = pd.concat(long_list, axis=1)

        if start_dt is not None:
            X = X.loc[X.index.get_level_values(0) >= start_dt]

        if y is not None:
            y_long = y.stack().rename("target")
            if start_dt is not None:
                y_long = y_long.loc[y_long.index.get_level_values(0) >= start_dt]
            df = pd.concat([X, y_long], axis=1)
        else:
            df = X
        return df

    def _per_ts_weighted_spearman(self, true_series, pred_series):
        df = pd.concat([true_series.rename("y"), pred_series.rename("p")], axis=1).dropna()
        if df.empty:
            return np.nan
        scores = []
        for _, g in df.groupby(level=0):
            y = g["y"].values; p = g["p"].values
            scores.append(self.weighted_spearmanr(y, p))
        return float(np.nanmean(scores)) if scores else np.nan

    """TRAIN"""
    def train(self, df_target, df_vwap, symbols):
        import gc, os
        self._log("[train] preparing labels & groups ...")
        y_long, group_sizes, group_times = self._build_y_and_groups(
            df_target, self.start_datetime, df_vwap.index, max(self.windows_main)
        )
        self._log(f"[train] samples={len(y_long):,}  timestamps={len(group_sizes):,}  avg_group={np.mean(group_sizes):.1f}")

        """Relevance + weights"""
        self._log("[train] computing relevance and weights ...")
        rel = np.empty(len(y_long), dtype=np.int32)
        w_extreme = np.empty(len(y_long), dtype=np.float32)
        offsets = np.cumsum([0] + list(group_sizes[:-1]))
        for t, gsz, off in zip(group_times, group_sizes, offsets):
            sl = slice(off, off + gsz)
            yb = y_long.iloc[sl].to_numpy()
            rel[sl] = pd.Series(yb).rank(ascending=True, method="first").astype(np.int32).to_numpy() - 1
            r_true = pd.Series(yb).rank(ascending=False, method="average")
            x = 2 * (r_true - 1) / (gsz - 1) - 1
            w_extreme[sl] = (x.values.astype(np.float32)) ** 2

        weights = w_extreme
        if self.use_time_decay:
            tmax = group_times.max()
            ages_sec = ((tmax - group_times) / np.timedelta64(1, 's')).astype(np.float64)
            hl_sec = float(self.halflife_days * 24 * 3600)
            decay_ts = np.exp(-np.log(2.0) * (ages_sec / hl_sec)).astype(np.float32)
            weights = (w_extreme * np.repeat(decay_ts, group_sizes)).astype(np.float32)

        """Build X on disk (memmap)"""
        F = len(self.windows_main) * 8 + 4
        X_path = "/kaggle/working/X_mm.dat"
        if os.path.exists(X_path):
            os.remove(X_path)
        X_mm = np.memmap(X_path, mode="w+", dtype=np.float32, shape=(len(y_long), F))

        self._log("[train] precomputing EMA panels ...")
        ema_panels = self._precompute_ema_panels(df_vwap)

        self._log("[train] filling window features (per-ts) ...")
        next_col = self._fill_memmap_features(X_mm, 0, df_vwap, ema_panels, y_long, group_sizes, group_times, symbols)
        self._log("[train] adding seasonality ...")
        next_col = self._fill_memmap_seasonality(X_mm, next_col, y_long)
        assert next_col == F

        """LightGBM datasets (set max_bin at Dataset creation)"""
        self._log("[train] building LightGBM datasets ...")
        dvalid = None
        dataset_params = {"max_bin": 63}  # memory-friendly binning

        if self.use_early_stopping:
            cut_ts = group_times.max() - np.timedelta64(self.valid_last_days, 'D')
            mask_ts = group_times < cut_ts
            if (mask_ts.sum() == 0) or (mask_ts.sum() == len(group_times)):
                k = max(1, int(0.9 * len(group_times)))
                cut_ts = group_times[k]
                mask_ts = group_times < cut_ts

            tr_rows = int(np.sum(group_sizes[mask_ts]))
            tr_mask = np.zeros(len(y_long), dtype=bool); tr_mask[:tr_rows] = True
            va_mask = ~tr_mask

            dtrain = lgb.Dataset(
                X_mm[tr_mask, :],
                label=rel[tr_mask],
                group=group_sizes[mask_ts].tolist(),
                weight=weights[tr_mask],
                free_raw_data=True,
                params=dataset_params,
            )
            dvalid = lgb.Dataset(
                X_mm[va_mask, :],
                label=rel[va_mask],
                group=group_sizes[~mask_ts].tolist(),
                weight=weights[va_mask],
                free_raw_data=True,
                params=dataset_params,
                reference=dtrain,
            )

        else:
            dtrain = lgb.Dataset(
                X_mm,
                label=rel,
                group=group_sizes.tolist(),
                weight=weights,
                free_raw_data=True,
                params=dataset_params,
            )

        params = dict(
            objective="lambdarank",
            metric="ndcg",
            ndcg_eval_at=[5, 10, 20],
            learning_rate=self.lr,
            num_leaves=63,
            feature_fraction=0.9,
            bagging_fraction=0.8,
            bagging_freq=1,
            min_data_in_leaf=50,
            lambda_l2=1.0,
            verbose=-1,
            seed=42,
            num_threads=self.num_threads,
        )

        params["label_gain"] = list(range(int(max(group_sizes)) + 1))

        self._log("[train] training LightGBM ...")
        callbacks = []
        if self.VERBOSE_PROGRESS:
            callbacks.append(lgb.log_evaluation(200))
        if dvalid is not None:
            callbacks.append(lgb.early_stopping(self.es_rounds, verbose=False))

        model = lgb.train(
            params,
            dtrain,
            num_boost_round=self.num_boost_round,
            valid_sets=[dtrain] if dvalid is None else [dtrain, dvalid],
            valid_names=["tr"] if dvalid is None else ["tr", "va"],
            callbacks=callbacks,
        )

        del X_mm, rel, weights, dtrain, dvalid, ema_panels
        gc.collect()

        self._log("[train] streaming submit.csv & check.csv ...")
        ema_panels = self._precompute_ema_panels(df_vwap)
        vwap_arr = df_vwap.to_numpy(copy=False); idx = df_vwap.index
        sym2col = {s: i for i, s in enumerate(symbols)}; N = len(symbols)

        def features_at_time(dt, syms_subset):
            tpos = idx.get_loc(dt)
            vt = vwap_arr[tpos, :]
            blocks = []
            for w in self.windows_main:
                vtm = vwap_arr[tpos - w, :]
                ema_t = ema_panels[w].to_numpy(copy=False)[tpos, :]
                mom_all = vt / vtm - 1.0
                eg_all = vt / ema_t - 1.0
                m_mom = np.nanmean(mom_all); s_mom = np.nanstd(mom_all, ddof=1); s_mom = s_mom if np.isfinite(s_mom) and s_mom != 0 else np.nan
                m_eg = np.nanmean(eg_all); s_eg = np.nanstd(eg_all, ddof=1); s_eg = s_eg if np.isfinite(s_eg) and s_eg != 0 else np.nan
                r_mom_all = pd.Series(mom_all).rank(ascending=False, method="average").to_numpy() / float(N)
                r_eg_all = pd.Series(eg_all).rank(ascending=False, method="average").to_numpy() / float(N)
                cols_idx = np.fromiter((sym2col[s] for s in syms_subset), count=len(syms_subset), dtype=np.int32)
                mom = mom_all[cols_idx]; eg = eg_all[cols_idx]
                momdm = mom - m_mom; egdm = eg - m_eg
                momz = (mom - m_mom) / s_mom if np.isfinite(s_mom) else np.zeros_like(mom)
                egz = (eg - m_eg) / s_eg if np.isfinite(s_eg) else np.zeros_like(eg)
                r_mom = r_mom_all[cols_idx]; r_eg = r_eg_all[cols_idx]
                blocks.append(np.stack([mom, momdm, momz, eg, egdm, egz, r_mom, r_eg], axis=1).astype(np.float32))
            Xw = np.concatenate(blocks, axis=1)
            h = dt.hour; d = dt.dayofweek
            seas = np.array([np.sin(2*np.pi*h/24.0), np.cos(2*np.pi*h/24.0),
                             np.sin(2*np.pi*d/7.0), np.cos(2*np.pi*d/7.0)], dtype=np.float32)
            return np.concatenate([Xw, np.broadcast_to(seas, (len(syms_subset), 4))], axis=1)

        try:
            _, sid_path = self._load_submission_id()
            self._log(f"[train] loaded submission_id from: {sid_path}")
        except:
            sid_path = "/kaggle/working/_tmp_submission_id_empty.csv"
            pd.DataFrame({"id": []}).to_csv(sid_path, index=False)

        chunksize = 200_000
        first_submit = True
        first_check = True

        for sid_chunk in pd.read_csv(sid_path, usecols=["id"], chunksize=chunksize):
            # --- NEW: make chunk indices 0..len-1 so grp.index matches pred_out ---
            sid_chunk = sid_chunk.reset_index(drop=True)

            parts = sid_chunk["id"].str.rsplit("_", n=1, expand=True)
            dt_chunk = pd.to_datetime(parts[0], errors="coerce")
            sym_chunk = parts[1].astype(str)
            sid_chunk["datetime"] = dt_chunk
            sid_chunk["symbol"]   = sym_chunk

            pred_out = np.zeros(len(sid_chunk), dtype=np.float32)

            for dt_val, grp in sid_chunk.groupby("datetime", sort=False):
                if pd.isna(dt_val) or (dt_val not in idx):
                    continue
                syms = grp["symbol"].astype(str).to_numpy()
                X_dt = features_at_time(dt_val, syms)
                pred_out[grp.index.to_numpy()] = model.predict(X_dt).astype(np.float32)

            out = sid_chunk[["id"]].copy()
            out["predict_return"] = pred_out
            out.to_csv("submit.csv", index=False, header=first_submit, mode=("w" if first_submit else "a"))
            first_submit = False

            dtu = sid_chunk["datetime"].dropna().unique()
            y_slice = df_target.loc[df_target.index.intersection(dtu)]
            if not y_slice.empty:
                dft = y_slice.stack().rename("true_return").reset_index()
                dft.columns = ["datetime", "symbol", "true_return"]
                merged = sid_chunk.merge(dft, how="left", on=["datetime", "symbol"])[["id", "true_return"]].dropna(subset=["true_return"])
                if not merged.empty:
                    merged["true_return"] = merged["true_return"].astype(np.float32)
                    merged.to_csv("check.csv", index=False, header=first_check, mode=("w" if first_check else "a"))
                    first_check = False

            del sid_chunk, parts, dt_chunk, sym_chunk, pred_out
            if 'y_slice' in locals(): del y_slice
            gc.collect()


        self._log("[train] streamed writing complete: submit.csv, check.csv")

    def run(self):
        self._log("=== RUN START ===")
        t0 = self._tic()

        (symbols, time_arr,
         open_price_arr, high_price_arr, low_price_arr,
         close_price_arr, vwap_arr, amount_arr) = self.get_all_symbol_kline()

        idx = pd.DatetimeIndex(time_arr)
        df_vwap = pd.DataFrame(vwap_arr, index=idx, columns=symbols).astype(np.float32)

        del open_price_arr, high_price_arr, low_price_arr, close_price_arr, vwap_arr, amount_arr
        gc.collect()

        W1D = 4*24
        df_24h_rtn = df_vwap / df_vwap.shift(W1D) - 1.0
        df_target  = df_24h_rtn.shift(-W1D)

        self.train(df_target, df_vwap, symbols)

        self._toc(t0, "TOTAL run")
        self._log("=== RUN END ===")


In [ ]:
if __name__ == '__main__':
    model = OlsModel()
    model.run()

In [ ]:
import pandas as pd
row_count = len(pd.read_csv("check.csv"))
print(f"Number of rows: {row_count}")
pd.read_csv("check.csv", nrows = 100)

In [ ]:
import pandas as pd
row_count = len(pd.read_csv("submit.csv"))
print(f"Number of rows: {row_count}")
pd.read_csv("submit.csv", nrows = 100)

In [ ]:
import pandas as pd
row_count = len(pd.read_csv("/kaggle/input/avenir-hku-web/submission_id.csv"))
print(f"Number of rows: {row_count}")
pd.read_csv("/kaggle/input/avenir-hku-web/submission_id.csv", nrows = 100)

In [ ]:
from IPython.display import FileLink
print("\nDownload your submission file below:")
display(FileLink("submit.csv"))